In [1]:
from lokigi.site import SiteProblem
import geopandas
import pandas as pd
import pickle
import imageio.v2 as imageio
from pathlib import Path
import matplotlib.pyplot as plt
import io
from PIL import Image
import plotly.express as px

C:\geographic_or_ds_playground\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
problem = SiteProblem()

problem.add_demand(
    pd.read_csv("demand_MF_50_84.csv"), demand_col="MF50-84", location_id_col="LSOA 2021 Name"
)

problem.add_region_geometry_layer(
    geopandas.read_file("LSOA_Devon_2021_EW_BSC_V4.gpkg"), common_col="LSOA21NM"
)

problem.add_equity_data("devon_imd_2025_2021_LSOAs.csv",
                        equity_col="Index of Multiple Deprivation (IMD) Decile (where 1 is most deprived 10% of LSOA",
                        common_col="LSOA name (2021)", disadvantaged_end="low", label="IMD Decile")

existing_cdcs = pd.read_csv("devon_cdcs.csv")
existing_cdcs_gdf =     geopandas.GeoDataFrame(
        existing_cdcs,  # Our pandas dataframe
        geometry=geopandas.points_from_xy(
            existing_cdcs[
                "Longitude"
            ],  # Our 'x' column (horizontal position of points)
            existing_cdcs["Latitude"],  # Our 'y' column (vertical position of points)
        ),
        crs="EPSG:4326",
    )

problem.add_sites(
        existing_cdcs_gdf,
        candidate_id_col="Facility_Name",
        required_sites_col="Existing"
    )

problem_car = problem.copy()

problem_car.add_travel_matrix(
    pd.read_csv("travel_matrix_car.csv"), unit="minutes", source_col="from_id"
)

# problem_car.add_additional_data(
#     pd.read_csv("travel_matrix_public_transport.csv").fillna(9999.0), common_col="from_id", label="Public Transport Travel Time (minutes)",
#     column_of_interest=""
# )

problem_pt = problem.copy()

problem_pt.add_travel_matrix(
    pd.read_csv("travel_matrix_public_transport.csv"), unit="minutes", source_col="from_id",
    allow_missing=True, treat_as_missing=9999,
)

# problem_pt.add_additional_data(
#     pd.read_csv("travel_matrix_car.csv").fillna(9999.0), common_col="from_id", label="Car Travel Time (minutes)"
# )

In [3]:
solutions_car = []

for i in range(4, 16):
    solutions_car.append({'n': i, 'solution': problem_car.solve(
        p=i,
        threshold_for_coverage=30,
        rank_on="proportion_demand_improved",
        baseline=True,
        meaningful_change_threshold=5.0,
        beyond_thresholds=[45, 60],
        n_jobs=-1,
    )})

  0%|          | 0/1 [00:00<?, ?it/s]

100%|██████████| 1/1 [00:06<00:00,  6.09s/it]

100%|██████████| 1/1 [00:06<00:00,  6.09s/it]

  0%|          | 0/14 [00:00<?, ?it/s]

  7%|▋         | 1/14 [00:10<02:15, 10.44s/it]

...

 48%|████▊     | 175/364 [00:02<00:02, 87.99it/s]

 56%|█████▋    | 205/364 [00:04<00:03, 45.66it/s]

 73%|███████▎  | 265/364 [00:04<00:01, 78.22it/s]

 91%|█████████ | 330/364 [00:05<00:00, 87.05it/s]

100%|██████████| 364/364 [00:05<00:00, 70.38it/s]

### Compare with best results for 1, 2 and 3 sites

We have to set these up slightly differently because this version of lokigi can't handle p being < number of sites marked as 'required'. 

In [4]:
problem_car_simple = problem_car.copy()

problem_car_simple.add_sites(
        existing_cdcs_gdf[existing_cdcs_gdf["Existing"]=="Yes"],
        candidate_id_col="Facility_Name",
    )

problem_pt_simple = problem_pt.copy()

problem_pt_simple.add_sites(
        existing_cdcs_gdf[existing_cdcs_gdf["Existing"]=="Yes"],
        candidate_id_col="Facility_Name",
    )

In [5]:
solution_1_car = problem_car_simple.solve(p=1, threshold_for_coverage=30)
solution_2_car = problem_car_simple.solve(p=2, threshold_for_coverage=30)
solution_3_car = problem_car_simple.solve(p=3, threshold_for_coverage=30)

  0%|          | 0/4 [00:00<?, ?it/s]

100%|██████████| 4/4 [00:00<00:00, 16.46it/s]

100%|██████████| 4/4 [00:00<00:00, 16.40it/s]

  0%|          | 0/6 [00:00<?, ?it/s]

100%|██████████| 6/6 [00:00<00:00, 14.32it/s]

100%|██████████| 6/6 [00:00<00:00, 14.27it/s]

  0%|          | 0/4 [00:00<?, ?it/s]

100%|██████████| 4/4 [00:00<00:00, 15.08it/s]

100%|██████████| 4/4 [00:00<00:00, 14.95it/s]

In [6]:
solutions_car.append({'n': 1, 'solution':solution_1_car})
solutions_car.append({'n': 2, 'solution':solution_2_car})
solutions_car.append({'n': 3, 'solution':solution_3_car})

# Compare best

In [7]:
result_df_comparison = []

for sol in solutions_car:
    result_df_comparison.append(sol['solution'].return_best_combination_details(top_n=1))

result_df_comparison = pd.concat(result_df_comparison)
result_df_comparison['n'] = result_df_comparison['site_indices'].apply(lambda x: len(x))
result_df_comparison

,index,solution_rank,site_names,site_indices,unselected_site_names,coverage_threshold,weighted_average,unweighted_average,90th_percentile,max,...,sites_added_vs_baseline,demand_beyond_threshold_45,regions_beyond_threshold_45,demand_beyond_threshold_45_by_equity_group,regions_beyond_threshold_45_by_equity_group,demand_beyond_threshold_60,regions_beyond_threshold_60,demand_beyond_threshold_60_by_equity_group,regions_beyond_threshold_60_by_equity_group,n
0,0,1,"[Bideford Community Hospital, NHS Nightingale ...","[0, 1, 2, 3]","[Tiverton - Lowman Way, Okehampton - Exeter Ro...",30,23.392076,21.421445,42.396666,61.533333,...,[],52145.0,58.0,"{1: 658.0, 2: 1169.0, 3: 2402.0, 4: 16403.0, 5...","{1: 1, 2: 2, 3: 3, 4: 17, 5: 19, 6: 8, 7: 5, 8...",2049.0,2.0,"{1: 0.0, 2: 0.0, 3: 0.0, 4: 942.0, 5: 0.0, 6: ...","{1: 0, 2: 0, 3: 0, 4: 1, 5: 0, 6: 1, 7: 0, 8: ...",4
0,0,1,"[Bideford Community Hospital, NHS Nightingale ...","[0, 1, 2, 3, 8]","[Tiverton - Lowman Way, Okehampton - Exeter Ro...",30,22.147958,20.206196,42.073333,61.183334,...,[Newton Abbott - Market Walk],49306.0,55.0,"{1: 658.0, 2: 1169.0, 3: 2402.0, 4: 16403.0, 5...","{1: 1, 2: 2, 3: 3, 4: 17, 5: 17, 6: 8, 7: 4, 8...",942.0,1.0,"{1: 0.0, 2: 0.0, 3: 0.0, 4: 942.0, 5: 0.0, 6: ...","{1: 0, 2: 0, 3: 0, 4: 1, 5: 0, 6: 0, 7: 0, 8: ...",5
0,0,1,"[Bideford Community Hospital, NHS Nightingale ...","[0, 1, 2, 3, 8, 12]","[Tiverton - Lowman Way, Okehampton - Exeter Ro...",30,20.689798,18.811545,38.690000,61.183334,...,"[Newton Abbott - Market Walk, Barnstaple - Arc...",36959.0,40.0,"{1: 0.0, 2: 0.0, 3: 1893.0, 4: 10068.0, 5: 124...","{1: 0, 2: 0, 3: 2, 4: 11, 5: 14, 6: 6, 7: 4, 8...",942.0,1.0,"{1: 0.0, 2: 0.0, 3: 0.0, 4: 942.0, 5: 0.0, 6: ...","{1: 0, 2: 0, 3: 0, 4: 1, 5: 0, 6: 0, 7: 0, 8: ...",6
0,0,1,"[Bideford Community Hospital, NHS Nightingale ...","[0, 1, 2, 3, 8, 12, 17]","[Tiverton - Lowman Way, Okehampton - Exeter Ro...",30,19.744424,17.906516,37.170000,61.183334,...,"[Newton Abbott - Market Walk, Barnstaple - Arc...",32446.0,36.0,"{1: 0.0, 2: 0.0, 3: 1893.0, 4: 10068.0, 5: 885...","{1: 0, 2: 0, 3: 2, 4: 11, 5: 11, 6: 6, 7: 3, 8...",942.0,1.0,"{1: 0.0, 2: 0.0, 3: 0.0, 4: 942.0, 5: 0.0, 6: ...","{1: 0, 2: 0, 3: 0, 4: 1, 5: 0, 6: 0, 7: 0, 8: ...",7
0,0,1,"[Bideford Community Hospital, NHS Nightingale ...","[0, 1, 2, 3, 8, 10, 12, 17]","[Tiverton - Lowman Way, Okehampton - Exeter Ro...",30,18.674391,16.994810,33.549999,61.183334,...,"[Newton Abbott - Market Walk, Honiton - Devons...",22237.0,24.0,"{1: 0.0, 2: 0.0, 3: 1893.0, 4: 9359.0, 5: 5991...","{1: 0, 2: 0, 3: 2, 4: 10, 5: 7, 6: 4, 7: 1, 8:...",942.0,1.0,"{1: 0.0, 2: 0.0, 3: 0.0, 4: 942.0, 5: 0.0, 6: ...","{1: 0, 2: 0, 3: 0, 4: 1, 5: 0, 6: 0, 7: 0, 8: ...",8
0,0,1,"[Bideford Community Hospital, NHS Nightingale ...","[0, 1, 2, 3, 4, 8, 10, 12, 17]","[Okehampton - Exeter Road Industrial Estate, C...",30,17.609237,16.055578,29.900000,61.183334,...,"[Tiverton - Lowman Way, Newton Abbott - Market...",18435.0,20.0,"{1: 0.0, 2: 0.0, 3: 982.0, 4: 9359.0, 5: 4186....","{1: 0, 2: 0, 3: 1, 4: 10, 5: 5, 6: 3, 7: 1, 8:...",942.0,1.0,"{1: 0.0, 2: 0.0, 3: 0.0, 4: 942.0, 5: 0.0, 6: ...","{1: 0, 2: 0, 3: 0, 4: 1, 5: 0, 6: 0, 7: 0, 8: ...",9
0,0,1,"[Bideford Community Hospital, NHS Nightingale ...","[0, 1, 2, 3, 4, 8, 10, 12, 15, 17]","[Okehampton - Exeter Road Industrial Estate, C...",30,16.501467,15.173548,28.583334,53.633335,...,"[Tiverton - Lowman Way, Newton Abbott - Market...",9521.0,10.0,"{1: 0.0, 2: 0.0, 3: 982.0, 4: 2909.0, 5: 1722....","{1: 0, 2: 0, 3: 1, 4: 3, 5: 2, 6: 3, 7: 1, 8: ...",0.0,0.0,"{1: 0.0, 2: 0.0, 3: 0.0, 4: 0.0, 5: 0.0, 6: 0....","{1: 0, 2: 0, 3: 0, 4: 0, 5: 0, 6: 0, 7: 0, 8: ...",10
0,0,1,"[Bideford Community Hospital, NHS Nightingale ...","[0, 1, 2, 3, 4, 8, 10, 11, 12, 15, 17]","[Okehampton - Exeter Road Industrial Estate, C...",30,15.848246,14.603292,28.583334,53.633335,...,"[Tiverton - Lowman Way, Newton Abbott - Market...",9521.0,10.0,"{1: 0.0, 2: 0.0, 3: 982.0, 4: 2909.0, 5: 1722....","{1: 0, 2: 0, 3: 1, 4: 3, 5

In [8]:
result_df_comparison.to_pickle("comparison_num_sites.pkl")

In [9]:
px.bar(result_df_comparison, x="n", y="weighted_average")

In [10]:
px.bar(result_df_comparison, x="n", y="max")

In [11]:
px.bar(result_df_comparison, x="n", y="90th_percentile")

## Explore solutions

In [12]:
solution_car_5 = [i for i in solutions_car if i["n"] == 5][0]['solution']

In [13]:
solution_car_5.show_solutions()

,solution_rank,site_names,site_indices,unselected_site_names,coverage_threshold,weighted_average,unweighted_average,90th_percentile,max,weighted_average_for_ranking,...,sites_closed_vs_baseline,sites_added_vs_baseline,demand_beyond_threshold_45,regions_beyond_threshold_45,demand_beyond_threshold_45_by_equity_group,regions_beyond_threshold_45_by_equity_group,demand_beyond_threshold_60,regions_beyond_threshold_60,demand_beyond_threshold_60_by_equity_group,regions_beyond_threshold_60_by_equity_group
0,1,"[Bideford Community Hospital, NHS Nightingale ...","[0, 1, 2, 3, 8]","[Tiverton - Lowman Way, Okehampton - Exeter Ro...",30,22.15,20.21,42.07,61.18,22.15,...,[],[Newton Abbott - Market Walk],49306.0,55,"{1: 658.0, 2: 1169.0, 3: 2402.0, 4: 16403.0, 5...","{1: 1, 2: 2, 3: 3, 4: 17, 5: 17, 6: 8, 7: 4, 8...",942.0,1,"{1: 0.0, 2: 0.0, 3: 0.0, 4: 942.0, 5: 0.0, 6: ...","{1: 0, 2: 0, 3: 0, 4: 1, 5: 0, 6: 0, 7: 0, 8: ..."
1,2,"[Bideford Community Hospital, NHS Nightingale ...","[0, 1, 2, 3, 17]","[Tiverton - Lowman Way, Okehampton - Exeter Ro...",30,22.26,20.37,41.91,61.18,22.26,...,[],[Paignton - Hyde Road],45487.0,52,"{1: 658.0, 2: 1169.0, 3: 2402.0, 4: 16403.0, 5...","{1: 1, 2: 2, 3: 3, 4: 17, 5: 15, 6: 8, 7: 3, 8...",942.0,1,"{1: 0.0, 2: 0.0, 3: 0.0, 4: 942.0, 5: 0.0, 6: ...","{1: 0, 2: 0, 3: 0, 4: 1, 5: 0, 6: 0, 7: 0, 8: ..."
2,3,"[Bideford Community Hospital, NHS Nightingale ...","[0, 1, 2, 3, 12]","[Tiverton - Lowman Way, Okehampton - Exeter Ro...",30,21.93,20.03,39.41,61.53,21.93,...,[],[Barnstaple - Archwood Retail Park],39798.0,43,"{1: 0.0, 2: 0.0, 3: 1893.0, 4: 10068.0, 5: 142...","{1: 0, 2: 0, 3: 2, 4: 11, 5: 16, 6: 6, 7: 5, 8...",2049.0,2,"{1: 0.0, 2: 0.0, 3: 0.0, 4: 942.0, 5: 0.0, 6: ...","{1: 0, 2: 0, 3: 0, 4: 1, 5: 0, 6: 1, 7: 0, 8: ..."
3,4,"[Bideford Community Hospital, NHS Nightingale ...","[0, 1, 2, 3, 9]","[Tiverton - Lowman Way, Okehampton - Exeter Ro...",30,22.26,20.38,41.27,61.53,22.26,...,[],[Bovey Tracey - Blue Waters],46019.0,51,"{1: 658.0, 2: 1169.0, 3: 2402.0, 4: 13656.0, 5...","{1: 1, 2: 2, 3: 3, 4: 14, 5: 15, 6: 8, 7: 5, 8...",2049.0,2,"{1: 0.0, 2: 0.0, 3: 0.0, 4: 942.0, 5: 0.0, 6: ...","{1: 0, 2: 0, 3: 0, 4: 1, 5: 0, 6: 1, 7: 0, 8: ..."
4,5,"[Bideford Community Hospital, NHS Nightingale ...","[0, 1, 2, 3, 10]","[Tiverton - Lowman Way, Okehampton - Exeter Ro...",30,22.32,20.51,39.44,61.53,22.32,...,[],[Honiton - Devonshire Road],41936.0,46,"{1: 658.0, 2: 1169.0, 3: 2402.0, 4: 15694.0, 5...","{1: 1, 2: 2, 3: 3, 4: 16, 5: 15, 6: 6, 7: 3, 8...",2049.0,2,"{1: 0.0, 2: 0.0, 3: 0.0, 4: 942.0, 5: 0.0, 6: ...","{1: 0, 2: 0, 3: 0, 4: 1, 5: 0, 6: 1, 7: 0, 8: ..."
5,6,"[Bideford Community Hospital, NHS Nightingale ...","[0, 1, 2, 3, 4]","[Okehampton - Exeter Road Industrial Estate, C...",30,22.19,20.39,41.07,61.53,22.19,...,[],[Tiverton - Lowman Way],44511.0,51,"{1: 658.0, 2: 1169.0, 3: 1491.0, 4: 12571.0, 5...","{1: 1, 2: 2, 3: 2, 4: 14, 5: 17, 6: 7, 7: 5, 8...",2049.0,2,"{1: 0.0, 2: 0.0, 3: 0.0, 4: 942.0, 5: 0.0, 6: ...","{1: 0, 2: 0, 3: 0, 4: 1, 5: 0, 6: 1, 7: 0, 8: ..."
6,7,"[Bideford Community Hospital, NHS Nightingale ...","[0, 1, 2, 3, 5]","[Tiverton - Lowman Way, Crediton - Lords Meado...",30,22.05,20.37,37.61,61.53,22.05,...,[],[Okehampton - Exeter Road Industrial Estate],34564.0,39,"{1: 658.0, 2: 1169.0, 3: 509.0, 4: 6816.0, 5: ...","{1: 1, 2: 2, 3: 1, 4: 7, 5: 13, 6: 7, 7: 5, 8:...",1107.0,1,"{1: 0.0, 2: 0.0, 3: 0.0, 4: 0.0, 5: 0.0, 6: 11...","{1: 0, 2: 0, 3: 0, 4: 0, 5: 0, 6: 1, 7: 0, 8: ..."
7,8,"[Bideford Community Hospital, NHS Nightingale ...","[0, 1, 2, 3, 7]","[Tiverton - Lowman Way, Okehampton - Exeter Ro...",30,22.42,20.69,39.84,61.53,22.42,...,[],[South Molton - Pathfields],35114.0,38,"{1: 0.0, 2: 0.0, 3: 982.0, 4: 10068.0, 5: 1157...","{1: 0, 2: 0, 3: 1, 4: 11, 5: 13, 6: 5, 7: 5, 8...",2049.0,2,"{1: 0.0, 2: 0.0, 3: 0.0, 4: 942.0, 5: 0.0, 6: ...","{1: 0, 2: 0, 3: 0, 4: 1, 5: 0, 6: 1, 7: 0, 8: ..."
8,9,"[Bideford Community Hospital, NHS Nightingale ...","[0, 1, 2, 3, 6]","[Tiverton - Lowman Way, Oke

In [14]:
solution_car_5.show_solutions_colnames()

Index(['solution_rank', 'site_names', 'site_indices', 'unselected_site_names',
       'coverage_threshold', 'weighted_average', 'unweighted_average',
       '90th_percentile', 'max', 'weighted_average_for_ranking',
       'unweighted_average_for_ranking', 'max_for_ranking', 'total_cost',
       'proportion_within_coverage_threshold',
       'proportion_regions_within_coverage_threshold',
       'demand_within_coverage_threshold', 'regions_within_coverage_threshold',
       'regions_unreachable', 'demand_unreachable',
       'proportion_demand_unreachable', 'weighted_by_equity_group',
       'unweighted_by_equity_group', 'coverage_by_equity_group',
       'coverage_regions_by_equity_group', 'max_cost_by_equity_group',
       'regions_unreachable_by_equity_group',
       'demand_unreachable_by_equity_group', 'gap_absolute_weighted',
       'gap_relative_weighted', 'avg_lower_third_bins',
       'avg_middle_third_bins', 'avg_upper_third_bins', 'inter_tertile_ratio',
       'gap_absolute_d

## Solution Outputs

In [15]:
solution_car_5 = [i for i in solutions_car if i["n"] == 5][0]['solution']

In [16]:
solution_car_5

In [17]:
with open("solution_car_5.pkl", "wb") as f:
    pickle.dump(solution_car_5, f)

In [18]:
solution_car_5.solution_df.to_pickle("solution_car_5_solution_df.pkl")

In [19]:
def generate_solution_animation(solution, output_filename, frame_duration=0.5):

    frames = []

    total_n_solutions = len(solution.solution_df)

    # Get existing sites
    sites = solution.site_problem.show_sites()
    existing_sites = sites[sites[solution.site_problem._candidate_sites_required_sites_col]=="Yes"][solution.site_problem._candidate_sites_candidate_id_col].to_list()

    for i in range(1, total_n_solutions + 1):
        # Work out which the one additional site is
        sites_in_solution = solution.solution_df[solution.solution_df["solution_rank"]==i]["site_names"].iloc[0]
        new_site = [i for i in sites_in_solution if i not in existing_sites]
        ax = solution.plot_best_combination(solution_rank=i, title=f"Trying out {new_site[0]}\n(Option {i} of {total_n_solutions})")
        fig = ax.figure

        buffer = io.BytesIO()
        fig.savefig(buffer, format="png", bbox_inches="tight")
        buffer.seek(0)

        frames.append(Image.open(buffer).copy())

        plt.close(fig)

    frames[0].save(
        f"{output_filename}.gif",
        save_all=True,
        append_images=frames[1:],
        duration=int(frame_duration * 1000),
        loop=0,
    )

In [20]:
solution_car_6 = [i for i in solutions_car if i["n"] == 6][0]['solution']

In [21]:
with open("solution_car_6.pkl", "wb") as f:
    pickle.dump(solution_car_6, f)

In [22]:
solution_car_6.solution_df.to_pickle("solution_car_6_solution_df.pkl")

## Animation Generation

In [23]:
generate_solution_animation(solution_car_5, output_filename="solution_car_5", frame_duration=0.8)

In [24]:
generate_solution_animation(solution_car_6, output_filename="solution_car_6", frame_duration=0.2)

In [25]:
# solution_car_7 = [i for i in solutions_car if i["n"] == 7][0]['solution']
# generate_solution_animation(solution_car_7, output_filename="solution_car_7", frame_duration=0.1)

In [26]:
# solution_car_8 = [i for i in solutions_car if i["n"] == 8][0]['solution']
# generate_solution_animation(solution_car_8, output_filename="solution_car_8", frame_duration=0.05)

In [27]:
# solution_car_6.solution_df.head(10).to_pickle("solution_car_6_best.pkl")
# solution_car_7.solution_df.head(10).to_pickle("solution_car_7_best.pkl")
# solution_car_8.solution_df.head(10).to_pickle("solution_car_8_best.pkl")

In [28]:
# solution_car_6.solution_df.tail(1).to_pickle("solution_car_6_worst.pkl")
# solution_car_7.solution_df.tail(1).to_pickle("solution_car_7_worst.pkl")
# solution_car_8.solution_df.tail(1).to_pickle("solution_car_8_worst.pkl")